# EACL 2027: Personalization-Placement Ablation — Judge-Condition Comparison

This notebook compares the **same 150 agent runs** scored under three different judge conditions,
so we can see whether the persona **leak fix** and the **rubric patch** change the study's conclusions.

| condition | output dir | what the judge saw |
|---|---|---|
| **old (leaky persona judge)** | `outputs/placement_ablation_v1` | full persona incl. `latent_profile` answer key; ignored the rubric |
| **rubric (leak-free)** | `outputs/placement_ablation_v1_rubric` | only the visible query + frozen rubric (no persona) |
| **rubric + latent_profile** | `outputs/placement_ablation_v1_rubric_profile` | rubric **plus** the persona answer key (A/B control) |

The agent runs are identical across conditions; only the judge input differs. `old` was produced by the
pre-fix judge; the two `rubric*` sets are produced by the leak-free judges against the patched rubrics.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

PROJECT_ROOT = os.path.dirname(os.getcwd())
OUT = os.path.join(PROJECT_ROOT, "outputs")

# label -> output dir for each judge condition we compare
CONDITIONS = {
    "old (leaky)":            "placement_ablation_v1",
    "rubric (leak-free)":     "placement_ablation_v1_rubric",
    "rubric + latent_profile":"placement_ablation_v1_rubric_profile",
}
COND_ORDER = list(CONDITIONS)
COND_COLORS = {
    "old (leaky)": "#bbbbbb",
    "rubric (leak-free)": "#4477aa",
    "rubric + latent_profile": "#ee6677",
}
VARIANT_ORDER = ["V0_generic_single","V1_generic_fanout","V2_synthesis_only_personalization",
                 "V3_personalized_fanout","V4_mixed_fanout"]

def _csv(dirname, name):
    p = os.path.join(OUT, dirname, name)
    return pd.read_csv(p) if os.path.exists(p) else None

def load_condition(dirname):
    return {
        "variant":           _csv(dirname, "summary_by_variant.csv"),
        "variant_task_type": _csv(dirname, "summary_by_variant_task_type.csv"),
        "contrasts":         _csv(dirname, "contrasts_by_task_type.csv"),
    }

DATA = {label: load_condition(d) for label, d in CONDITIONS.items()}
for label in COND_ORDER:
    v = DATA[label]["variant"]
    print(f"{label:26s} summary_by_variant: {None if v is None else v.shape}")

def combined(table_key):
    """Stack a table across all available conditions, adding a `condition` column."""
    frames = []
    for label in COND_ORDER:
        df = DATA[label][table_key]
        if df is None:
            continue
        df = df.copy(); df["condition"] = label
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

## Section 1 — Overall metrics, by judge condition

Grouped bars: each variant's mean for a metric, with one bar per judge condition. Gaps between the grey
(`old`) and blue (`rubric`) bars show how much the **leak fix** moved the score; gaps between blue and red
(`+latent_profile`) isolate the effect of re-introducing the answer key.

In [ ]:
key_metrics = ["final_intent_satisfaction", "final_personalization_target_use",
               "final_overpersonalization", "retrieval_result_persona_fit"]
cv = combined("variant")
fig, axes = plt.subplots(2, 2, figsize=(16, 11)); axes = axes.flatten()
for i, m in enumerate(key_metrics):
    col = f"{m}_mean"; ax = axes[i]
    if cv.empty or col not in cv.columns:
        ax.set_title(f"{m} (no data yet)"); ax.set_axis_off(); continue
    sns.barplot(data=cv, x="variant", y=col, hue="condition", hue_order=COND_ORDER,
                order=VARIANT_ORDER, palette=COND_COLORS, edgecolor="#666", linewidth=0.8, ax=ax)
    ax.set_title(m.replace("_", " ").title() + ("  (lower=better)" if "overpersonalization" in m else ""),
                 fontsize=13, fontweight="bold")
    ax.set_ylim(1, 5.2); ax.set_xlabel(""); ax.set_ylabel("Score (1-5)")
    ax.tick_params(axis="x", rotation=18); ax.legend(title="judge", fontsize=8)
plt.tight_layout(); plt.show()

## Section 2 — Task-type sensitivity, by judge condition

The core hypothesis lives here: does personalized **fan-out** (V3) help *retrieval-sensitive* tasks more,
and personalized **synthesis** (V2) help *synthesis-sensitive* tasks more — and does that pattern survive
the leak fix?

In [ ]:
metrics = ["final_intent_satisfaction", "final_overpersonalization"]
ctt = combined("variant_task_type")
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for r, tt in enumerate(["retrieval_sensitive", "synthesis_sensitive"]):
    sub = ctt[ctt["task_type"] == tt] if not ctt.empty else ctt
    for c, m in enumerate(metrics):
        col = f"{m}_mean"; ax = axes[r, c]
        if ctt.empty or col not in sub.columns:
            ax.set_title("no data yet"); ax.set_axis_off(); continue
        sns.barplot(data=sub, x="variant", y=col, hue="condition", hue_order=COND_ORDER,
                    order=VARIANT_ORDER, palette=COND_COLORS, edgecolor="#666", linewidth=0.8, ax=ax)
        ax.set_title(f"{tt}: {m.replace('_',' ').title()}" + ("  (lower=better)" if "overpers" in m else ""),
                     fontsize=12, fontweight="bold")
        ax.set_ylim(1, 5.2); ax.set_xlabel(""); ax.set_ylabel("Score (1-5)")
        ax.tick_params(axis="x", rotation=18); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## Section 3 — Do the headline contrasts change?

The paper's claims are **contrasts**: V2−V1 (synthesis personalization), V3−V2 (marginal fan-out),
V3−V1 (full personalization), V4−V3 (mixed fan-out). If the leak inflated/flipped any of these, the grey
bars will diverge from the blue. This is the empirical answer to "did the leak matter?".

In [ ]:
focus = "final_intent_satisfaction"
cc = combined("contrasts")
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for i, tt in enumerate(["retrieval_sensitive", "synthesis_sensitive"]):
    s = cc[(cc["metric"] == focus) & (cc["task_type"] == tt)] if not cc.empty else cc
    ax = axes[i]
    if cc.empty or s.empty:
        ax.set_title(f"{tt} (no data yet)"); ax.set_axis_off(); continue
    sns.barplot(data=s, y="contrast_name", x="diff_a_minus_b", hue="condition", hue_order=COND_ORDER,
                palette=COND_COLORS, edgecolor="#666", linewidth=0.8, ax=ax)
    ax.axvline(0.0, color="#888", ls="--", lw=1.2)
    ax.set_title(f"{focus} contrasts — {tt}", fontsize=12, fontweight="bold")
    ax.set_xlabel("mean_a - mean_b"); ax.set_ylabel(""); ax.legend(title="judge", fontsize=8)
plt.tight_layout(); plt.show()

### Contrast tables (the numbers behind the bars)

`Δ = mean_a − mean_b` for each contrast, one column per judge condition. Compare columns left-to-right:
large shifts between `old` and `rubric (leak-free)` mean the leak was distorting that conclusion.

In [ ]:
def contrast_table(metric):
    cc = combined("contrasts")
    if cc.empty: return cc
    s = cc[cc["metric"] == metric]
    if s.empty: return s
    t = s.pivot_table(index=["task_type", "contrast_name"], columns="condition",
                      values="diff_a_minus_b", aggfunc="first")
    return t.reindex(columns=[c for c in COND_ORDER if c in t.columns])

for metric in ["final_intent_satisfaction", "final_personalization_target_use", "final_overpersonalization"]:
    print(f"\nΔ (mean_a - mean_b) for {metric}" + ("  (lower=better)" if "overpers" in metric else ""))
    t = contrast_table(metric)
    try:
        from IPython.display import display
        display(t)
    except Exception:
        print(t)

## Section 4 — How to read this comparison

This notebook intentionally **does not hard-code conclusions** (the previous version cited fixed numbers —
e.g. "groundedness 4.27", "93% of full performance" — that were produced by the *leaky* judge and are no
longer trustworthy). Instead, read the conclusions off the three-condition comparison above:

1. **Did the leak fix change the ranking of variants?** Compare grey vs blue in Sections 1–2. If the
   best variant per task type is the same, the qualitative story holds; only the magnitudes were inflated.
2. **Did the leak fix change the headline contrasts?** Section 3 is decisive. A contrast that is large and
   positive under `old` but near-zero under `rubric (leak-free)` was a leak artifact.
3. **Does the judge need the answer key?** Compare blue (`rubric`) vs red (`rubric + latent_profile`). If
   they're close, the rubric carries all the signal the judge needs and the leak-free judge is sufficient.
   If red systematically inflates the gaps, that confirms the answer key was doing the work — i.e. the leak.

Whatever the rubric (leak-free) column says is the **trustworthy** result to report in the paper.